## <a name="feature_maps">3. backbone网络</a>

这一部分我们介绍Mask R-CNN的backbone网络。在Mask R-CNN中，backbone网络主要用于图像的特征提取。经过前面的学习，我们已经对图像的特征图有了一定了解，在Mask R-CNN中，使用了ResNet作为卷积网络，并结合FPN(Feature Pyramid Network，特征图金字塔）来生成图像的特征图。

### ResNet残差网络

在神经网络中，网络的深度是非常重要的。一般认为，随着深度的增加，神经网络可以提取到更加抽象、更加语义化的信息。随着神经网络深度的增加，也确实在一些问题上获得了更大的突破。但是，人们发现，直接在神经网络中堆叠层，会导致[梯度弥散和梯度爆炸问题](https://en.wikipedia.org/wiki/Vanishing_gradient_problem)。通过初始化和中间层的正则化(normalization)，可以解决梯度弥散和爆炸的问题，使得几十层的网络在使用反向传播算法时，梯度可以开始收敛。然而，随着层数的增加，网络出现了**退化**现象：精度出现饱和并且迅速下降。实验证明，退化问题并不是过拟合造成的，因为即使在训练集上，增加层数也会导致精度下降。退化问题表明，深层神经网络不能通过简单的增加深度进行优化。

#### 残差的概念和表示

既然简单叠加层的深度网络存在退化问题，那么，我们不妨使用一种方法，让若干层映射为一个层，这样，深层网络就会等同于浅层网络，从而避免退化问题。如何将多个层映射为一个层？直观上考虑，我们可以直接让若干个层去拟合一个潜在的恒等映射函数$H(x) = x$，这样，深层网络就可以退化为一个浅层网络，在这种思路下，映射的关系如下图所示：


![straightforward_identity_mapping](img/straightforward_identity_mapping.jpg)


但是，直接把多个层拟合为一个映射函数$H(x)$非常困难，因此，在论文《Deep Residual Learning for Image Recognition》中，作者引入了一种深度残差学习框架（deep residual learning framework），把输出结果设计为$H(x) = F(x) + x$，这样原有的若干层就变为一个残差函数$F(x) := H(x) - x$，只要$F(x) = 0$，就构成了一个恒等映射$H(x) = x$，如下图所示：

![residual_identity_mapping](img/residual_identity_mapping.jpg)

这样，拟合残差函数$F(x)$就比直接拟合恒等映射函数$H(x)$容易多了。按照上面的描述，identy部分为恒等映射(identity mapping)，也就是输入参数$x$本身，$F(x)$为残差映射（residual mapping)，因为$F(x)$为最终结果与恒等映射的差。在网络训练过程中，如果已经达到最优解，则残差部分向$0$拟合，只剩下恒等映射，这样，保证网络处于最优状态，而不会随着深度的增加导致性能下降。

#### shortcut的两种形式

恒等映射的部分，称为shortcut connection或skip connection。在ResNet中，我们统一用shortcut来表示。在ResNet中，shortcut可以直接等于输入参数$x$，用下面的公式表示：

$$y = F(x, {W_i}) + x$$

其中用$W_i$表示要学习的参数，为了便与表达，这里忽略了bias。此外，残差映射可能导致输入参数维度的变化，在做Add操作时，要求恒等映射与$F(x)$的维度一致，因此需要在恒等映射中加入线性映射(linear projection)，满足维度一致的要求，如下面的公式所示：

$$y = F(x, {W_i}) + W_sx$$

其中$W_s$表示我们在恒等映射中加入的线性映射，在实际编程中，我们一般用卷积层来控制恒等映射的维度。此外，在残差部分的处理中，作者在卷积和激活函数之间加入了BatchNorm层以优化网络的训练效果。这些内容在后面的代码实践中，会有相应的示例。

#### 残差学习的效果

论文中，作者基于VGG-19网络设计了一个34层的普通深层网络，并针对这个网络加入了残差学习进行对比，得到VGG-19，34-layer plain和34-layer residual三个网络。在ImageNet classification、CIFAR-10以及PASCAL、MS COCO的目标检测中，进行了实验和对比，ResNet都获得了更好的效果。具体结果可以参看论文，这里不再赘述。


#### ResNet代码实践

##### 在Keras中实现一个恒等层

在Keras中，我们通过Layer的叠加来实现深层网络，每个Layer执行特定的一组数据处理动作，如卷积、池化、激活函数等。此外，Keras还以Layer的形式提供运算操作，如多维数组的加法、乘法等。在实现恒等映射时，我们需要确保输入数据和处理后的数据在维度上满足运算操作Layer的要求。例如，在下面的例子中，我们将输入数据和处理后的数据进行加法运算，要求my_activation与input这两层的输出数据shape一致，或者满足[numpy broadcast](https://docs.scipy.org/doc/numpy/user/basics.broadcasting.html)要求。

In [ ]:
import keras.layers as KL
from keras.models import Model
from keras.utils import plot_model

input_shape = [256, 256, 3] # 假设输入为长宽都为256的RGB图片
input_depth = input_shape[2] # 获得图片的深度（RGB三个颜色通道，深度为3）


input_layer = KL.Input(shape=input_shape, name='input')

# 这里我们使用与input相同depth，尺寸为(1x1)的卷积核，卷积之后，shape不会发生变化
x = KL.Conv2D(input_depth, (1, 1), name='my_conv2d')(input_layer)
# 经过(1x1)的池化，输出数据的shape依然不会改变
x = KL.MaxPool2D((1, 1), name='my_maxpooling2d')(x)
x = KL.Activation('relu', name='my_activation')(x)

# 残差部分和恒等部分的输出shape保持一致
print('shapes of the two operands for Add:', x.shape, input_layer.shape)
x = KL.Add(name='my_add')([x, input_layer])
identity_model = Model(inputs=input_layer, outputs=x)

plot_model(identity_model, to_file='sample_identity_block.png')


![sample_identity_block](./sample_identity_block.png)

通过上面的示例，我们就使用Layer完成了一个恒等映射。接下来，我们构建一个真正的残差网络。


##### 定义residual block

在ResNet的构建中，我们要通过反复叠加若干个相同的残差模块来实现若干个stage。下面的代码定义了两种block，以便在后面的ResNet网络构建中调用。

在两个block函数中，我们可以看到，input_tensor都经过若干个Conv2D，BatchNorm和ReLU计算后，得到x，然后与shortcut进行Add操作。不同的是，identity_block中shortcut直接就是input_tensor，而conv_block的shortcut是input_tensor再经过Conv2D和BatchNorm后得到的。这两种block就实现了前面提到的两种不同形式的shortcut connection。

In [ ]:
def identity_block(input_tensor, kernel_size, filters, stage, block,
                   use_bias=True, train_bn=True):
    """The identity_block is the block that has no conv layer at shortcut
    # Arguments
        input_tensor: input tensor
        kernel_size: default 3, the kernel size of middle conv layer at main path
        filters: list of integers, the nb_filters of 3 conv layer at main path
        stage: integer, current stage label, used for generating layer names
        block: 'a','b'..., current block label, used for generating layer names
        use_bias: Boolean. To use or not use a bias in conv layers.
        train_bn: Boolean. Train or freeze Batch Norm layers
    """
    nb_filter1, nb_filter2, nb_filter3 = filters
    conv_name_base = 'res' + str(stage) + block + '_branch'
    bn_name_base = 'bn' + str(stage) + block + '_branch'

    x = KL.Conv2D(nb_filter1, (1, 1), name=conv_name_base + '2a',
                  use_bias=use_bias)(input_tensor)
    x = KL.BatchNormalization(name=bn_name_base + '2a')(x, training=train_bn)
    x = KL.Activation('relu')(x)

    x = KL.Conv2D(nb_filter2, (kernel_size, kernel_size), padding='same',
                  name=conv_name_base + '2b', use_bias=use_bias)(x)
    x = KL.BatchNormalization(name=bn_name_base + '2b')(x, training=train_bn)
    x = KL.Activation('relu')(x)

    x = KL.Conv2D(nb_filter3, (1, 1), name=conv_name_base + '2c',
                  use_bias=use_bias)(x)
    x = KL.BatchNormalization(name=bn_name_base + '2c')(x, training=train_bn)

    x = KL.Add()([x, input_tensor])
    x = KL.Activation('relu', name='res' + str(stage) + block + '_out')(x)
    return x

def conv_block(input_tensor, kernel_size, filters, stage, block,
               strides=(2, 2), use_bias=True, train_bn=True):
    """conv_block is the block that has a conv layer at shortcut
    # Arguments
        input_tensor: input tensor
        kernel_size: default 3, the kernel size of middle conv layer at main path
        filters: list of integers, the nb_filters of 3 conv layer at main path
        stage: integer, current stage label, used for generating layer names
        block: 'a','b'..., current block label, used for generating layer names
        use_bias: Boolean. To use or not use a bias in conv layers.
        train_bn: Boolean. Train or freeze Batch Norm layers
    Note that from stage 3, the first conv layer at main path is with subsample=(2,2)
    And the shortcut should have subsample=(2,2) as well
    """
    nb_filter1, nb_filter2, nb_filter3 = filters
    conv_name_base = 'res' + str(stage) + block + '_branch'
    bn_name_base = 'bn' + str(stage) + block + '_branch'

    x = KL.Conv2D(nb_filter1, (1, 1), strides=strides,
                  name=conv_name_base + '2a', use_bias=use_bias)(input_tensor)
    x = KL.BatchNormalization(name=bn_name_base + '2a')(x, training=train_bn)
    x = KL.Activation('relu')(x)

    x = KL.Conv2D(nb_filter2, (kernel_size, kernel_size), padding='same',
                  name=conv_name_base + '2b', use_bias=use_bias)(x)
    x = KL.BatchNormalization(name=bn_name_base + '2b')(x, training=train_bn)
    x = KL.Activation('relu')(x)

    x = KL.Conv2D(nb_filter3, (1, 1), name=conv_name_base +
                  '2c', use_bias=use_bias)(x)
    x = KL.BatchNormalization(name=bn_name_base + '2c')(x, training=train_bn)

    shortcut = KL.Conv2D(nb_filter3, (1, 1), strides=strides,
                         name=conv_name_base + '1', use_bias=use_bias)(input_tensor)
    shortcut = KL.BatchNormalization(name=bn_name_base + '1')(shortcut, training=train_bn)

    x = KL.Add()([x, shortcut])
    x = KL.Activation('relu', name='res' + str(stage) + block + '_out')(x)
    return x

##### 构建ResNet50网络

In [ ]:
def resnet_graph(input_layer, train_bn=False):
    # Stage 1
    x = KL.ZeroPadding2D((3, 3))(input_layer)
    x = KL.Conv2D(64, (7, 7), strides=(2, 2), name='conv1', use_bias=True)(x)
    x = KL.BatchNormalization(name='bn_conv1')(x, training=train_bn)
    x = KL.Activation('relu')(x)
    C1 = x = KL.MaxPooling2D((3, 3), strides=(2, 2), padding="same")(x)

    # Stage 2
    x = conv_block(x, 3, [64, 64, 256], stage=2, block='a', strides=(1, 1), train_bn=train_bn)
    x = identity_block(x, 3, [64, 64, 256], stage=2, block='b', train_bn=train_bn)
    C2 = x = identity_block(x, 3, [64, 64, 256], stage=2, block='c', train_bn=train_bn)
    # Stage 3
    x = conv_block(x, 3, [128, 128, 512], stage=3, block='a', train_bn=train_bn)
    x = identity_block(x, 3, [128, 128, 512], stage=3, block='b', train_bn=train_bn)
    x = identity_block(x, 3, [128, 128, 512], stage=3, block='c', train_bn=train_bn)
    C3 = x = identity_block(x, 3, [128, 128, 512], stage=3, block='d', train_bn=train_bn)

    # Stage 4
    x = conv_block(x, 3, [256, 256, 1024], stage=4, block='a', train_bn=train_bn)
    block_count = 5 # 5 for ResNet50
    for i in range(block_count):
        x = identity_block(x, 3, [256, 256, 1024], stage=4, block=chr(98 + i), train_bn=train_bn)
    C4 = x

    # Stage 5
    x = conv_block(x, 3, [512, 512, 2048], stage=5, block='a', train_bn=train_bn)
    x = identity_block(x, 3, [512, 512, 2048], stage=5, block='b', train_bn=train_bn)
    C5 = x = identity_block(x, 3, [512, 512, 2048], stage=5, block='c', train_bn=train_bn)
    return C1, C2, C3, C4, C5

# 首先，我们定义一个输入层
input_layer = KL.Input(shape=(1024, 1024, 3), name='input_layer')
C1, C2, C3, C4, C5 = resnet_graph(input_layer)

print(C1)
print(C2)
print(C3)
print(C4)
print(C5)

通过ResNet构造代码可以看到，从stage2开始，每个stage都有1个conv_block和若干个identity_block，每个block所采用的卷积核尺寸都是(3x3)，但是每个stage的输出数据的深度逐步增加，又因为卷及操作是叠加的，所以每个stage输出数据的尺寸越来越小，这个过程可以从上面输出的C1到C5的shape得到验证。

以图片来理解每个stage输出的shape，随着卷积层的增多，图片的尺寸越来越小，而深度则不断增加。下面我们载入一张图片作为输入，经过ResNet的计算，查看每个stage得到的输出。

In [ ]:
import numpy as np
import tensorflow as tf
from PIL import Image
import matplotlib.pyplot as plt

%matplotlib inline

src_img = Image.open('./data/dog.jpg')
img = np.array(src_img)
# 进行运算时，必须为输入数据加上一个维度，表示batch
batch_img = np.expand_dims(img, axis=0)

input_img = KL.Input(shape=img.shape, name='input_dog_img')
resnet_stages = resnet_graph(input_img)

with tf.Session() as sess:
    sess.run(tf.global_variables_initializer())
    current_stage = 1
    for stage in resnet_stages:
        result = sess.run(stage, feed_dict={input_img: batch_img})
        # 去掉batch的维度，得到本stage计算的结果，shape为(width, height, depth)
        unwind_result = np.squeeze(result)
        feature_indexes = np.random.choice(unwind_result.shape[2], 4)
        num_subplot = 1
        for index in feature_indexes:
            feature = unwind_result[:, :, index]
            width, height = feature.shape[:2]
            plt.subplot(2, 2, num_subplot)
            num_subplot += 1
            plt.title('{}x{}@stage{}'.format(width, height, current_stage))
            plt.axis('off')
            plt.imshow(feature)
        plt.show()
        current_stage += 1

### FPN

相关介绍，图像金字塔，特征金字塔，如何结合ResNet生成FPN

代码实践

通过ResNet和FPN，我们构建了backbone网络，输入一张图片，并通过backbone网络对其进行处理，获得5个level的特征图